# Tiling visualization
> Viz utils for tiling visualization

In [ ]:
#| default_exp viz_utils.tiling_viz

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| export
import sys
from pathlib import Path


In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))


In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))


In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from dotenv import load_dotenv
from typing import List, Tuple, Optional, Union
import torch
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
import torch.nn.functional as F



In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/homes/hasan/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/homes/hasan/projects/git_data/new_test/nbs'

In [ ]:
#| export
def visualize_tile_overlay(
    original_image: np.ndarray,
    tiles: Union[torch.Tensor, np.ndarray],
    positions: List[Tuple],
    tile_size: int = 256,
    overlap: int = 32,
    figsize: Tuple[int, int] = (20, 15),
    save_path: Optional[Union[str, Path]] = None,
    show_grid: bool = True,
    alpha: float = 0.3,
    colormap: str = 'tab20'
) -> None:
    """
    Visualize how tiles overlay on the original image with comprehensive display.
    
    Args:
        original_image: Original image as numpy array [H, W] or [H, W, C]
        tiles: Tensor or array of tiles [N, C, tile_size, tile_size] or [N, tile_size, tile_size]
        positions: List of (row, col, start_h, start_w) for each tile
        tile_size: Size of each tile
        overlap: Overlap between tiles
        figsize: Figure size for the plot
        save_path: Path to save the visualization
        show_grid: Whether to show grid lines
        alpha: Transparency for overlay
        colormap: Colormap for different tiles
    """
    
    # Convert tensors to numpy if needed
    if isinstance(tiles, torch.Tensor):
        tiles = tiles.cpu().numpy()
    
    # Handle different image formats
    if len(original_image.shape) == 3 and original_image.shape[2] == 3:
        display_image = original_image
    elif len(original_image.shape) == 3 and original_image.shape[2] == 1:
        display_image = original_image.squeeze(-1)
    else:
        display_image = original_image
    
    # Create figure with subplots
    fig = plt.figure(figsize=figsize)
    
    # Main layout: 2x2 grid
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], width_ratios=[2, 1])
    
    # 1. Original image with tile boundaries
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(display_image, cmap='gray' if len(display_image.shape) == 2 else None)
    ax1.set_title(f'Original Image with Tile Boundaries\n({len(positions)} tiles, size={tile_size}, overlap={overlap})', 
                  fontsize=14, fontweight='bold')
    
    # Create colormap for tiles
    colors = plt.cm.get_cmap(colormap)(np.linspace(0, 1, len(positions)))
    
    # Draw tile boundaries
    for i, (row, col, start_h, start_w) in enumerate(positions):
        rect = patches.Rectangle(
            (start_w, start_h), tile_size, tile_size,
            linewidth=2, edgecolor=colors[i], facecolor=colors[i], alpha=alpha,
            label=f'Tile {i}'
        )
        ax1.add_patch(rect)
        
        # Add tile number in center
        center_x = start_w + tile_size // 2
        center_y = start_h + tile_size // 2
        ax1.text(center_x, center_y, str(i), 
                ha='center', va='center', fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    if show_grid:
        ax1.grid(True, alpha=0.3)
    ax1.set_xlabel('Width (pixels)')
    ax1.set_ylabel('Height (pixels)')
    
    # 2. Tile positions heatmap
    ax2 = fig.add_subplot(gs[0, 1])
    tile_count_map = np.zeros_like(display_image[:, :, 0] if len(display_image.shape) == 3 else display_image)
    
    for i, (row, col, start_h, start_w) in enumerate(positions):
        end_h = min(start_h + tile_size, tile_count_map.shape[0])
        end_w = min(start_w + tile_size, tile_count_map.shape[1])
        tile_count_map[start_h:end_h, start_w:end_w] += 1
    
    im2 = ax2.imshow(tile_count_map, cmap='hot', interpolation='nearest')
    ax2.set_title('Tile Overlap Heatmap\n(Darker = More Overlap)', fontsize=12, fontweight='bold')
    plt.colorbar(im2, ax=ax2, label='Number of Overlapping Tiles')
    ax2.set_xlabel('Width (pixels)')
    ax2.set_ylabel('Height (pixels)')
    
    # 3. Sample tiles grid
    ax3 = fig.add_subplot(gs[1, :])
    n_sample_tiles = min(8, len(tiles))  # Show up to 8 sample tiles
    sample_indices = np.linspace(0, len(tiles)-1, n_sample_tiles, dtype=int)
    
    for idx, tile_idx in enumerate(sample_indices):
        ax_tile = plt.subplot(2, n_sample_tiles, n_sample_tiles + idx + 1)
        
        # Handle different tile formats
        if len(tiles[tile_idx].shape) == 3:
            tile_img = tiles[tile_idx][0] if tiles[tile_idx].shape[0] == 1 else tiles[tile_idx]
        else:
            tile_img = tiles[tile_idx]
        
        ax_tile.imshow(tile_img, cmap='gray')
        ax_tile.set_title(f'Tile {tile_idx}', fontsize=10)
        ax_tile.axis('off')
        
        # Add position info
        if tile_idx < len(positions):
            row, col, start_h, start_w = positions[tile_idx]
            ax_tile.text(0.5, -0.1, f'Pos: ({start_h},{start_w})', 
                        transform=ax_tile.transAxes, ha='center', fontsize=8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Tile overlay visualization saved to: {save_path}")
    
    plt.show()

In [ ]:
#| export
def visualize_tile_grid(
    tiles: Union[torch.Tensor, np.ndarray],
    positions: List[Tuple],
    max_tiles: int = 16,
    figsize: Tuple[int, int] = (16, 12),
    save_path: Optional[Union[str, Path]] = None
) -> None:
    """
    Visualize individual tiles in a grid layout.
    
    Args:
        tiles: Tensor or array of tiles
        positions: List of tile positions
        max_tiles: Maximum number of tiles to display
        figsize: Figure size
        save_path: Path to save the visualization
    """
    
    if isinstance(tiles, torch.Tensor):
        tiles = tiles.cpu().numpy()
    
    n_tiles = min(len(tiles), max_tiles)
    n_cols = int(np.ceil(np.sqrt(n_tiles)))
    n_rows = int(np.ceil(n_tiles / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1 or n_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    for i in range(n_tiles):
        # Handle different tile formats
        if len(tiles[i].shape) == 3:
            tile_img = tiles[i][0] if tiles[i].shape[0] == 1 else tiles[i]
        else:
            tile_img = tiles[i]
        
        axes[i].imshow(tile_img, cmap='gray')
        
        # Add detailed title with position info
        if i < len(positions):
            row, col, start_h, start_w = positions[i]
            title = f'Tile {i}\nRow:{row}, Col:{col}\nPos:({start_h},{start_w})'
        else:
            title = f'Tile {i}'
        
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    
    # Hide empty subplots
    for i in range(n_tiles, len(axes)):
        axes[i].axis('off')
        axes[i].set_visible(False)
    
    plt.suptitle(f'Individual Tiles Grid (Showing {n_tiles}/{len(tiles)} tiles)', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Tile grid visualization saved to: {save_path}")
    
    plt.show()

In [ ]:
#| export
def visualize_tile_positions(
    image_shape: Tuple[int, int],
    positions: List[Tuple],
    tile_size: int = 256,
    figsize: Tuple[int, int] = (12, 8),
    save_path: Optional[Union[str, Path]] = None
) -> None:
    """
    Visualize tile positions as a schematic diagram.
    
    Args:
        image_shape: Shape of the original image (H, W)
        positions: List of tile positions
        tile_size: Size of each tile
        figsize: Figure size
        save_path: Path to save the visualization
    """
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # 1. Tile position schematic
    H, W = image_shape
    colors = plt.cm.tab20(np.linspace(0, 1, len(positions)))
    
    ax1.set_xlim(0, W)
    ax1.set_ylim(H, 0)  # Invert y-axis to match image coordinates
    ax1.set_aspect('equal')
    
    for i, (row, col, start_h, start_w) in enumerate(positions):
        rect = patches.Rectangle(
            (start_w, start_h), tile_size, tile_size,
            linewidth=2, edgecolor=colors[i], facecolor=colors[i], alpha=0.3
        )
        ax1.add_patch(rect)
        
        # Add tile number
        center_x = start_w + tile_size // 2
        center_y = start_h + tile_size // 2
        ax1.text(center_x, center_y, str(i), 
                ha='center', va='center', fontsize=10, fontweight='bold')
    
    ax1.set_title(f'Tile Positions Schematic\n{len(positions)} tiles on {H}×{W} image', 
                  fontsize=12, fontweight='bold')
    ax1.set_xlabel('Width (pixels)')
    ax1.set_ylabel('Height (pixels)')
    ax1.grid(True, alpha=0.3)
    
    # 2. Position statistics
    ax2.axis('off')
    
    # Calculate statistics
    total_pixels = H * W
    tile_pixels = tile_size * tile_size
    coverage_pixels = len(positions) * tile_pixels
    overlap_ratio = (coverage_pixels - total_pixels) / total_pixels if coverage_pixels > total_pixels else 0
    
    stats_text = f"""
    📊 Tile Statistics:
    
    Image Size: {H} × {W} = {total_pixels:,} pixels
    Tile Size: {tile_size} × {tile_size} = {tile_pixels:,} pixels
    Number of Tiles: {len(positions)}
    
    Total Tile Pixels: {coverage_pixels:,}
    Overlap Ratio: {overlap_ratio:.2%}
    
    📍 Position Range:
    Min H: {min(pos[2] for pos in positions)}
    Max H: {max(pos[2] for pos in positions)}
    Min W: {min(pos[3] for pos in positions)}
    Max W: {max(pos[3] for pos in positions)}
    
    🔢 Grid Layout:
    Rows: {max(pos[0] for pos in positions) + 1}
    Cols: {max(pos[1] for pos in positions) + 1}
    """
    
    ax2.text(0.1, 0.9, stats_text, transform=ax2.transAxes, 
             fontsize=11, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Tile positions visualization saved to: {save_path}")
    
    plt.show()


In [ ]:
#| export
def create_tile_overlay_visualization(
    image_path: Union[str, Path],
    tile_processor,
    save_dir: Optional[Union[str, Path]] = None,
    show_all: bool = True
) -> None:
    """
    Create comprehensive tile overlay visualization from an image file.
    
    Args:
        image_path: Path to the image file
        tile_processor: TileProcessor instance
        save_dir: Directory to save visualizations
        show_all: Whether to show all visualization types
    """
    
    # Read image
    if isinstance(image_path, str):
        image_path = Path(image_path)
    
    # Try different image reading methods
    try:
        import cv2
        image = cv2.imread(str(image_path))
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    except:
        try:
            from PIL import Image
            image = np.array(Image.open(image_path))
        except:
            print(f"❌ Could not read image: {image_path}")
            return
    
    print(f"📸 Processing image: {image_path.name}")
    print(f"📏 Image shape: {image.shape}")
    
    # Convert to tensor for tile processing
    if len(image.shape) == 3:
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).float()
    else:
        image_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float()
    
    # Process tiles
    tiles, positions = tile_processor(image_tensor)
    
    print(f"🔪 Generated {tiles.shape[0]} tiles")
    print(f"📦 Tile shape: {tiles.shape[1:]}")
    
    # Create save directory if specified
    if save_dir:
        save_dir = Path(save_dir)
        save_dir.mkdir(exist_ok=True, parents=True)
        base_name = image_path.stem
    
    # Show visualizations
    if show_all:
        # 1. Main overlay visualization
        save_path1 = save_dir / f"{base_name}_tile_overlay.png" if save_dir else None
        visualize_tile_overlay(
            image, tiles, positions, 
            tile_size=tile_processor.tile_size,
            overlap=tile_processor.overlap,
            save_path=save_path1
        )
        
        # 2. Tile grid
        save_path2 = save_dir / f"{base_name}_tile_grid.png" if save_dir else None
        visualize_tile_grid(tiles, positions, save_path=save_path2)
        
        # 3. Position schematic
        save_path3 = save_dir / f"{base_name}_tile_positions.png" if save_dir else None
        visualize_tile_positions(
            image.shape[:2], positions, 
            tile_size=tile_processor.tile_size,
            save_path=save_path3
        )
    
    print(f"✅ Tile overlay visualization complete!") 

In [ ]:
#| export
def visualize_tile_mask_overlay(
    original_image: np.ndarray,
    tiles: Union[torch.Tensor, np.ndarray],
    tile_masks: Union[torch.Tensor, np.ndarray],
    positions: List[Tuple],
    tile_size: int = 256,
    overlap: int = 32,
    mask_threshold: float = 0.5,
    figsize: Tuple[int, int] = (20, 15),
    save_path: Optional[Union[str, Path]] = None,
    alpha: float = 0.6,
    mask_color: str = 'red',
    show_confidence: bool = True
) -> None:
    """
    Visualize mask predictions for each tile overlaid on the original image.
    
    Args:
        original_image: Original image as numpy array [H, W] or [H, W, C]
        tiles: Tensor or array of tiles [N, C, tile_size, tile_size]
        tile_masks: Predicted masks for each tile [N, C, tile_size, tile_size] or [N, tile_size, tile_size]
        positions: List of (row, col, start_h, start_w) for each tile
        tile_size: Size of each tile
        overlap: Overlap between tiles
        mask_threshold: Threshold for binary mask visualization
        figsize: Figure size for the plot
        save_path: Path to save the visualization
        alpha: Transparency for mask overlay
        mask_color: Color for mask overlay ('red', 'green', 'blue', 'yellow', etc.)
        show_confidence: Whether to show confidence values as intensity
    """
    
    # Convert tensors to numpy if needed
    if isinstance(tiles, torch.Tensor):
        tiles = tiles.cpu().numpy()
    if isinstance(tile_masks, torch.Tensor):
        tile_masks = tile_masks.cpu().numpy()
    
    # Handle different image formats
    if len(original_image.shape) == 3 and original_image.shape[2] == 3:
        display_image = original_image.copy()
    elif len(original_image.shape) == 3 and original_image.shape[2] == 1:
        display_image = np.repeat(original_image, 3, axis=2)
    else:
        display_image = np.stack([original_image] * 3, axis=2)
    
    # Ensure display_image is uint8
    if display_image.dtype != np.uint8:
        display_image = (display_image * 255).astype(np.uint8) if display_image.max() <= 1 else display_image.astype(np.uint8)
    
    # Handle different mask formats
    if len(tile_masks.shape) == 4 and tile_masks.shape[1] == 1:
        tile_masks = tile_masks[:, 0]  # Remove channel dimension if single channel
    elif len(tile_masks.shape) == 4:
        tile_masks = tile_masks[:, 0]  # Take first channel if multiple
    
    # Create figure with subplots
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 3, height_ratios=[2, 1], width_ratios=[2, 1, 1])
    
    # 1. Original image with tile mask overlays
    ax1 = fig.add_subplot(gs[0, 0])
    overlay_image = display_image.copy()
    
    # Color mapping for masks
    color_map = {
        'red': [255, 0, 0],
        'green': [0, 255, 0],
        'blue': [0, 0, 255],
        'yellow': [255, 255, 0],
        'cyan': [0, 255, 255],
        'magenta': [255, 0, 255]
    }
    mask_rgb = color_map.get(mask_color, [255, 0, 0])
    
    # Create combined mask overlay
    combined_mask = np.zeros(display_image.shape[:2], dtype=np.float32)
    confidence_map = np.zeros(display_image.shape[:2], dtype=np.float32)
    count_map = np.zeros(display_image.shape[:2], dtype=np.float32)
    
    for i, (row, col, start_h, start_w) in enumerate(positions):
        end_h = min(start_h + tile_size, display_image.shape[0])
        end_w = min(start_w + tile_size, display_image.shape[1])
        
        # Get tile mask
        tile_mask = tile_masks[i]
        
        # Resize mask if needed
        actual_h = end_h - start_h
        actual_w = end_w - start_w
        if tile_mask.shape != (actual_h, actual_w):
            tile_mask = cv2.resize(tile_mask, (actual_w, actual_h))
        
        # Add to combined mask
        if show_confidence:
            combined_mask[start_h:end_h, start_w:end_w] += tile_mask
            confidence_map[start_h:end_h, start_w:end_w] += tile_mask
        else:
            combined_mask[start_h:end_h, start_w:end_w] += (tile_mask > mask_threshold).astype(np.float32)
        
        count_map[start_h:end_h, start_w:end_w] += 1
    
    # Average overlapping regions
    count_map = np.maximum(count_map, 1)
    combined_mask = combined_mask / count_map
    
    if show_confidence:
        confidence_map = confidence_map / count_map
        # Use confidence as alpha
        mask_alpha = np.clip(confidence_map * alpha, 0, 1)
    else:
        # Binary mask
        combined_mask = (combined_mask > mask_threshold).astype(np.float32)
        mask_alpha = combined_mask * alpha
    
    # Apply mask overlay
    for c in range(3):
        overlay_image[:, :, c] = (
            overlay_image[:, :, c] * (1 - mask_alpha) + 
            mask_rgb[c] * mask_alpha
        ).astype(np.uint8)
    
    ax1.imshow(overlay_image)
    ax1.set_title(f'Original Image with Tile Mask Overlays\n({len(positions)} tiles, threshold={mask_threshold:.2f})', 
                  fontsize=14, fontweight='bold')
    ax1.set_xlabel('Width (pixels)')
    ax1.set_ylabel('Height (pixels)')
    
    # Add tile boundaries
    for i, (row, col, start_h, start_w) in enumerate(positions):
        rect = patches.Rectangle(
            (start_w, start_h), tile_size, tile_size,
            linewidth=1, edgecolor='white', facecolor='none', alpha=0.5
        )
        ax1.add_patch(rect)
    
    # 2. Confidence/probability heatmap
    ax2 = fig.add_subplot(gs[0, 1])
    if show_confidence:
        im2 = ax2.imshow(confidence_map, cmap='hot', vmin=0, vmax=1)
        ax2.set_title('Mask Confidence Heatmap', fontsize=12, fontweight='bold')
        plt.colorbar(im2, ax=ax2, label='Confidence')
    else:
        im2 = ax2.imshow(combined_mask, cmap='gray', vmin=0, vmax=1)
        ax2.set_title('Binary Mask', fontsize=12, fontweight='bold')
        plt.colorbar(im2, ax=ax2, label='Mask Value')
    
    ax2.set_xlabel('Width (pixels)')
    ax2.set_ylabel('Height (pixels)')
    
    # 3. Statistics
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axis('off')
    
    # Calculate statistics
    total_pixels = combined_mask.size
    if show_confidence:
        high_conf_pixels = np.sum(confidence_map > 0.7)
        med_conf_pixels = np.sum((confidence_map > 0.3) & (confidence_map <= 0.7))
        low_conf_pixels = np.sum(confidence_map <= 0.3)
        avg_confidence = np.mean(confidence_map)
        max_confidence = np.max(confidence_map)
    else:
        mask_pixels = np.sum(combined_mask > mask_threshold)
        mask_percentage = (mask_pixels / total_pixels) * 100
    
    overlap_pixels = np.sum(count_map > 1)
    overlap_percentage = (overlap_pixels / total_pixels) * 100
    
    if show_confidence:
        stats_text = f"""
📊 Mask Statistics:

🎯 Confidence Distribution:
High (>0.7): {high_conf_pixels:,} pixels
Medium (0.3-0.7): {med_conf_pixels:,} pixels  
Low (<0.3): {low_conf_pixels:,} pixels

📈 Confidence Metrics:
Average: {avg_confidence:.3f}
Maximum: {max_confidence:.3f}

🔄 Overlap Info:
Overlapping: {overlap_pixels:,} pixels
Overlap %: {overlap_percentage:.1f}%

🎨 Visualization:
Threshold: {mask_threshold:.2f}
Alpha: {alpha:.2f}
Color: {mask_color}
        """
    else:
        stats_text = f"""
📊 Mask Statistics:

🎯 Binary Mask:
Mask Pixels: {mask_pixels:,}
Total Pixels: {total_pixels:,}
Coverage: {mask_percentage:.1f}%

🔄 Overlap Info:
Overlapping: {overlap_pixels:,} pixels
Overlap %: {overlap_percentage:.1f}%

🎨 Visualization:
Threshold: {mask_threshold:.2f}
Alpha: {alpha:.2f}
Color: {mask_color}
        """
    
    ax3.text(0.05, 0.95, stats_text, transform=ax3.transAxes, 
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))
    
    # 4. Sample tile masks
    ax4 = fig.add_subplot(gs[1, :])
    n_sample_tiles = min(6, len(tiles))
    sample_indices = np.linspace(0, len(tiles)-1, n_sample_tiles, dtype=int)
    
    for idx, tile_idx in enumerate(sample_indices):
        ax_tile = plt.subplot(2, n_sample_tiles, n_sample_tiles + idx + 1)
        
        # Show tile with mask overlay
        if len(tiles[tile_idx].shape) == 3:
            tile_img = tiles[tile_idx][0] if tiles[tile_idx].shape[0] == 1 else tiles[tile_idx]
        else:
            tile_img = tiles[tile_idx]
        
        # Convert tile to RGB if needed
        if len(tile_img.shape) == 2:
            tile_display = np.stack([tile_img] * 3, axis=2)
        else:
            tile_display = tile_img
        
        # Normalize tile image
        if tile_display.max() <= 1:
            tile_display = (tile_display * 255).astype(np.uint8)
        else:
            tile_display = tile_display.astype(np.uint8)
        
        # Overlay mask on tile
        tile_mask = tile_masks[tile_idx]
        if show_confidence:
            mask_alpha_tile = np.clip(tile_mask * alpha, 0, 1)
        else:
            mask_alpha_tile = (tile_mask > mask_threshold).astype(np.float32) * alpha
        
        tile_overlay = tile_display.copy()
        for c in range(3):
            tile_overlay[:, :, c] = (
                tile_overlay[:, :, c] * (1 - mask_alpha_tile) + 
                mask_rgb[c] * mask_alpha_tile
            ).astype(np.uint8)
        
        ax_tile.imshow(tile_overlay)
        
        # Add title with statistics
        if show_confidence:
            avg_conf = np.mean(tile_mask)
            max_conf = np.max(tile_mask)
            title = f'Tile {tile_idx}\nAvg: {avg_conf:.2f}, Max: {max_conf:.2f}'
        else:
            mask_ratio = np.mean(tile_mask > mask_threshold)
            title = f'Tile {tile_idx}\nMask: {mask_ratio:.1%}'
        
        ax_tile.set_title(title, fontsize=9)
        ax_tile.axis('off')
        
        # Add position info
        if tile_idx < len(positions):
            row, col, start_h, start_w = positions[tile_idx]
            ax_tile.text(0.5, -0.15, f'Pos: ({start_h},{start_w})', 
                        transform=ax_tile.transAxes, ha='center', fontsize=8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Tile mask overlay visualization saved to: {save_path}")
    
    plt.show()


In [ ]:
#| export
def visualize_individual_tile_masks(
    tiles: Union[torch.Tensor, np.ndarray],
    tile_masks: Union[torch.Tensor, np.ndarray],
    positions: List[Tuple],
    max_tiles: int = 12,
    mask_threshold: float = 0.5,
    figsize: Tuple[int, int] = (16, 12),
    save_path: Optional[Union[str, Path]] = None,
    show_confidence: bool = True
) -> None:
    """
    Visualize individual tiles with their mask predictions.
    
    Args:
        tiles: Tensor or array of tiles
        tile_masks: Predicted masks for each tile
        positions: List of tile positions
        max_tiles: Maximum number of tiles to display
        mask_threshold: Threshold for binary mask visualization
        figsize: Figure size
        save_path: Path to save the visualization
        show_confidence: Whether to show confidence values
    """
    
    if isinstance(tiles, torch.Tensor):
        tiles = tiles.cpu().numpy()
    if isinstance(tile_masks, torch.Tensor):
        tile_masks = tile_masks.cpu().numpy()
    
    # Handle different mask formats
    if len(tile_masks.shape) == 4 and tile_masks.shape[1] == 1:
        tile_masks = tile_masks[:, 0]
    elif len(tile_masks.shape) == 4:
        tile_masks = tile_masks[:, 0]
    
    n_tiles = min(len(tiles), max_tiles)
    n_cols = int(np.ceil(np.sqrt(n_tiles)))
    n_rows = int(np.ceil(n_tiles / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols * 3, figsize=figsize)
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_tiles):
        row = i // n_cols
        col_base = (i % n_cols) * 3
        
        # Original tile
        if len(tiles[i].shape) == 3:
            tile_img = tiles[i][0] if tiles[i].shape[0] == 1 else tiles[i]
        else:
            tile_img = tiles[i]
        
        axes[row, col_base].imshow(tile_img, cmap='gray')
        axes[row, col_base].set_title(f'Tile {i}\nOriginal', fontsize=10)
        axes[row, col_base].axis('off')
        
        # Mask
        tile_mask = tile_masks[i]
        if show_confidence:
            im = axes[row, col_base + 1].imshow(tile_mask, cmap='hot', vmin=0, vmax=1)
            axes[row, col_base + 1].set_title(f'Confidence\nAvg: {np.mean(tile_mask):.2f}', fontsize=10)
        else:
            binary_mask = (tile_mask > mask_threshold).astype(np.float32)
            im = axes[row, col_base + 1].imshow(binary_mask, cmap='gray', vmin=0, vmax=1)
            mask_ratio = np.mean(binary_mask)
            axes[row, col_base + 1].set_title(f'Binary Mask\nCoverage: {mask_ratio:.1%}', fontsize=10)
        
        axes[row, col_base + 1].axis('off')
        
        # Overlay
        if len(tile_img.shape) == 2:
            tile_display = np.stack([tile_img] * 3, axis=2)
        else:
            tile_display = tile_img
        
        if tile_display.max() <= 1:
            tile_display = (tile_display * 255).astype(np.uint8)
        else:
            tile_display = tile_display.astype(np.uint8)
        
        # Create overlay
        if show_confidence:
            mask_alpha = np.clip(tile_mask * 0.6, 0, 1)
        else:
            mask_alpha = (tile_mask > mask_threshold).astype(np.float32) * 0.6
        
        tile_overlay = tile_display.copy()
        for c in range(3):
            tile_overlay[:, :, c] = (
                tile_overlay[:, :, c] * (1 - mask_alpha) + 
                255 * mask_alpha
            ).astype(np.uint8)
        
        axes[row, col_base + 2].imshow(tile_overlay)
        axes[row, col_base + 2].set_title('Overlay', fontsize=10)
        axes[row, col_base + 2].axis('off')
        
        # Add position info
        if i < len(positions):
            row_pos, col_pos, start_h, start_w = positions[i]
            axes[row, col_base].text(0.5, -0.1, f'Pos: ({start_h},{start_w})', 
                                   transform=axes[row, col_base].transAxes, 
                                   ha='center', fontsize=8)
    
    # Hide empty subplots
    total_subplots = n_rows * n_cols * 3
    used_subplots = n_tiles * 3
    for i in range(used_subplots, total_subplots):
        row = i // (n_cols * 3)
        col = i % (n_cols * 3)
        if row < n_rows and col < n_cols * 3:
            axes[row, col].axis('off')
            axes[row, col].set_visible(False)
    
    plt.suptitle(f'Individual Tile Masks (Showing {n_tiles}/{len(tiles)} tiles)', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Individual tile masks visualization saved to: {save_path}")
    
    plt.show()

In [ ]:
#| export
def visualize_merged_mask_comparison(
    original_image: np.ndarray,
    tile_masks: Union[torch.Tensor, np.ndarray],
    positions: List[Tuple],
    tile_size: int = 256,
    merged_mask: Optional[Union[torch.Tensor, np.ndarray]] = None,
    ground_truth: Optional[np.ndarray] = None,
    mask_threshold: float = 0.5,
    figsize: Tuple[int, int] = (18, 12),
    save_path: Optional[Union[str, Path]] = None
) -> None:
    """
    Compare individual tile masks with merged result and optionally ground truth.
    
    Args:
        original_image: Original image
        tile_masks: Individual tile mask predictions
        positions: Tile positions
        tile_size: Size of tiles
        merged_mask: Merged mask result (optional)
        ground_truth: Ground truth mask (optional)
        mask_threshold: Threshold for binary masks
        figsize: Figure size
        save_path: Path to save visualization
    """
    
    if isinstance(tile_masks, torch.Tensor):
        tile_masks = tile_masks.cpu().numpy()
    if merged_mask is not None and isinstance(merged_mask, torch.Tensor):
        merged_mask = merged_mask.cpu().numpy()
    
    # Handle different image formats
    if len(original_image.shape) == 3 and original_image.shape[2] == 3:
        display_image = original_image
    elif len(original_image.shape) == 3 and original_image.shape[2] == 1:
        display_image = original_image.squeeze(-1)
    else:
        display_image = original_image
    
    # Handle different mask formats
    if len(tile_masks.shape) == 4 and tile_masks.shape[1] == 1:
        tile_masks = tile_masks[:, 0]
    elif len(tile_masks.shape) == 4:
        tile_masks = tile_masks[:, 0]
    
    # Create combined mask from tiles
    combined_mask = np.zeros(display_image.shape[:2], dtype=np.float32)
    count_map = np.zeros(display_image.shape[:2], dtype=np.float32)
    
    for i, (row, col, start_h, start_w) in enumerate(positions):
        end_h = min(start_h + tile_size, display_image.shape[0])
        end_w = min(start_w + tile_size, display_image.shape[1])
        
        tile_mask = tile_masks[i]
        actual_h = end_h - start_h
        actual_w = end_w - start_w
        
        if tile_mask.shape != (actual_h, actual_w):
            tile_mask = cv2.resize(tile_mask, (actual_w, actual_h))
        
        combined_mask[start_h:end_h, start_w:end_w] += tile_mask
        count_map[start_h:end_h, start_w:end_w] += 1
    
    count_map = np.maximum(count_map, 1)
    combined_mask = combined_mask / count_map
    
    # Determine number of subplots
    n_plots = 2  # Original + Combined
    if merged_mask is not None:
        n_plots += 1
    if ground_truth is not None:
        n_plots += 1
    
    fig, axes = plt.subplots(2, n_plots, figsize=figsize)
    if n_plots == 1:
        axes = axes.reshape(2, 1)
    
    plot_idx = 0
    
    # Original image
    axes[0, plot_idx].imshow(display_image, cmap='gray' if len(display_image.shape) == 2 else None)
    axes[0, plot_idx].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0, plot_idx].axis('off')
    
    axes[1, plot_idx].imshow(display_image, cmap='gray' if len(display_image.shape) == 2 else None)
    axes[1, plot_idx].set_title('Original Image', fontsize=12)
    axes[1, plot_idx].axis('off')
    plot_idx += 1
    
    # Combined tile masks
    im1 = axes[0, plot_idx].imshow(combined_mask, cmap='hot', vmin=0, vmax=1)
    axes[0, plot_idx].set_title('Combined Tile Masks\n(Confidence)', fontsize=12, fontweight='bold')
    axes[0, plot_idx].axis('off')
    plt.colorbar(im1, ax=axes[0, plot_idx])
    
    binary_combined = (combined_mask > mask_threshold).astype(np.float32)
    axes[1, plot_idx].imshow(binary_combined, cmap='gray', vmin=0, vmax=1)
    coverage = np.mean(binary_combined) * 100
    axes[1, plot_idx].set_title(f'Binary Combined\n(Coverage: {coverage:.1f}%)', fontsize=12)
    axes[1, plot_idx].axis('off')
    plot_idx += 1
    
    # Merged mask (if provided)
    if merged_mask is not None:
        if len(merged_mask.shape) == 3:
            merged_mask = merged_mask.squeeze()
        
        im2 = axes[0, plot_idx].imshow(merged_mask, cmap='hot', vmin=0, vmax=1)
        axes[0, plot_idx].set_title('Merged Mask\n(Post-processed)', fontsize=12, fontweight='bold')
        axes[0, plot_idx].axis('off')
        plt.colorbar(im2, ax=axes[0, plot_idx])
        
        binary_merged = (merged_mask > mask_threshold).astype(np.float32)
        axes[1, plot_idx].imshow(binary_merged, cmap='gray', vmin=0, vmax=1)
        merged_coverage = np.mean(binary_merged) * 100
        axes[1, plot_idx].set_title(f'Binary Merged\n(Coverage: {merged_coverage:.1f}%)', fontsize=12)
        axes[1, plot_idx].axis('off')
        plot_idx += 1
    
    # Ground truth (if provided)
    if ground_truth is not None:
        axes[0, plot_idx].imshow(ground_truth, cmap='gray', vmin=0, vmax=1)
        axes[0, plot_idx].set_title('Ground Truth', fontsize=12, fontweight='bold')
        axes[0, plot_idx].axis('off')
        
        gt_coverage = np.mean(ground_truth) * 100
        axes[1, plot_idx].imshow(ground_truth, cmap='gray', vmin=0, vmax=1)
        axes[1, plot_idx].set_title(f'Ground Truth\n(Coverage: {gt_coverage:.1f}%)', fontsize=12)
        axes[1, plot_idx].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Merged mask comparison saved to: {save_path}")
    
    plt.show()

In [ ]:
def create_tile_mask_visualization(
    image_path: Union[str, Path],
    tile_processor,
    model,
    save_dir: Optional[Union[str, Path]] = None,
    mask_threshold: float = 0.5,
    show_all: bool = True,
    device: str = 'cpu'
) -> None:
    """
    Create comprehensive tile mask visualization from an image file and model.
    
    Args:
        image_path: Path to the image file
        tile_processor: TileProcessor instance
        model: Model for mask prediction
        save_dir: Directory to save visualizations
        mask_threshold: Threshold for binary masks
        show_all: Whether to show all visualization types
        device: Device for model inference
    """
    
    # Read image
    if isinstance(image_path, str):
        image_path = Path(image_path)
    
    try:
        import cv2
        image = cv2.imread(str(image_path))
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    except:
        try:
            from PIL import Image
            image = np.array(Image.open(image_path))
        except:
            print(f"❌ Could not read image: {image_path}")
            return
    
    print(f"📸 Processing image: {image_path.name}")
    print(f"📏 Image shape: {image.shape}")
    
    # Convert to tensor for tile processing
    if len(image.shape) == 3:
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).float()
    else:
        image_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float()
    
    # Process tiles
    tiles, positions = tile_processor(image_tensor)
    print(f"🔪 Generated {tiles.shape[0]} tiles")
    
    # Run model inference
    model.eval()
    with torch.no_grad():
        tiles_device = tiles.to(device)
        tile_masks = model(tiles_device)
        tile_masks = tile_masks.cpu()
    
    print(f"🤖 Generated mask predictions: {tile_masks.shape}")
    
    # Create save directory if specified
    if save_dir:
        save_dir = Path(save_dir)
        save_dir.mkdir(exist_ok=True, parents=True)
        base_name = image_path.stem
    
    # Show visualizations
    if show_all:
        # 1. Main tile mask overlay
        save_path1 = save_dir / f"{base_name}_tile_mask_overlay.png" if save_dir else None
        visualize_tile_mask_overlay(
            image, tiles, tile_masks, positions,
            tile_size=tile_processor.tile_size,
            overlap=tile_processor.overlap,
            mask_threshold=mask_threshold,
            save_path=save_path1
        )
        
        # 2. Individual tile masks
        save_path2 = save_dir / f"{base_name}_individual_tile_masks.png" if save_dir else None
        visualize_individual_tile_masks(
            tiles, tile_masks, positions,
            mask_threshold=mask_threshold,
            save_path=save_path2
        )
        
        # 3. Merged comparison (if tile merger available)
        try:
            from new_test.patching.preprocessing_input_image import TileMergerv4
            tile_merger = TileMergerv4()
            merged_mask = tile_merger(tile_masks, positions, image.shape[0], image.shape[1])
            
            save_path3 = save_dir / f"{base_name}_merged_comparison.png" if save_dir else None
            visualize_merged_mask_comparison(
                image, tile_masks, positions,
                tile_size=tile_processor.tile_size,
                merged_mask=merged_mask,
                mask_threshold=mask_threshold,
                save_path=save_path3
            )
        except:
            print("ℹ️  Tile merger not available, skipping merged comparison")
    
    print(f"✅ Tile mask visualization complete!") 

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('53_viz_utils.tiling_viz.ipynb')

ValueError: '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\nbs\\39_preprocessing.zero_degree_solder_pin.ipynb' is not in the subpath of '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\nbs'